In [ ]:
import fitz #PyMuPDF for reading PDF
from langchain_core.documents import documents
from transformers import CLIPProcessor, CLIPModel #used for converting text and images into vector embeddings, it is an opensource model trained on images and text. It is a combination of vision transformers and transformers (text transformers are called as transformers)
from PIL import Image
import torch
import numpy as np
from langchain.chat_models import init_chat_model
from langchain.prompts import PromptTemplate
from langchain.schema.messages import HumanMessage
from sklearn.metrics.pairwise import cosin_similarity
import os
import base64
import io 
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

In [ ]:
###CLip Model

import os
from dotenv import load_dotenv
load_dotenv()

#setting up the environment
os.environ["OPENAI_API_KEY"]=os.getenv("OPEN_API_KEY")

### initialize the CLIP model for unified embeddings
#variable clip_model, the model is called from hugging face. the model name is openai/clip-vit-base-patch32
clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
#clip_processor is the variable. It converts the input into the format the model requires
clip_processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model_eval()

In [ ]:
###Embedding Functions
def embed_image(image_data):
    """Embed image using CLIP"""
    if isinstance(image_data, str): #if Path
       image = Image.open(image_data).convert("RGB")
    else: #if PIL Image
        image = image_data
    #converting the format into tensors
    inputs = clip_processor(image=image,return_tensors="pt")
    with torch.no_grad():
        #built-in feature of CLIP model that will get the image features from the image input
        features = clip_model.get_image_features(**inputs)
        #normalize embeddings to unit vector
        features = features / features.norm(dim = -1, keepdim=True)
        return features.squeeze().numpy()

def embed_text(text):
    """Embed text using CLIP"""
    inputs = clip_processor
    (text=text,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=77 #CLIP's max token length 
    )
    with torch.no_grad():
    features = clip_model.get_text_features(**inputs)
    #normalize embeddings
    features = features / features.norm(dim=-1, keepdim=True)
    return features.squeeze().numpy()

In [ ]:
### Process PDF
pdf_path="multimodal_sample.pdf"
doc=fitz.open(pdf_path)
# Storage for all documents and embeddings
all_docs = []
all_embeddings = []
image_data_store = {} #store actual image data for the LLM

#Text splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

In [ ]:
for i,page in enumerate(doc):
    ##process text
    text=page.get_text()
    #remove empty spaces
    if text.strip():
        ##create temporary documents for splitting
        temp_doc = Document(page_content=text, metadata={"page": i, "type": "text"}) #for all the text content keep the metadata type as text
        text_chunks = splitter.split_documents([temp_doc])

        #Embed each chunk using CLI
        for chunk in text_chunks:
            embedding = embed_text(chunk.page_content) #calls the embed_text function, which will convert to vector and normalize the vector
            all_embeddings.append(embedding)
            all_doc.append(chunk)

    ##Process Images
    #Three Important Actions:

    #convert PDF image to PIL format
    #store as base64 for GPT-4V (which needs based64 images)
    #Create CLIP embeddings for retrieval

    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            #(converted the image information into image bytes)
            
            # Convert to PIL Image
            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
            
            # Create unique identifier
            image_id = f"page_{i}_img_{img_index}"
            
            # Store image as base64 for later use with GPT-4V
            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode()
            image_data_store[image_id] = img_base64
            
            # Embed image using CLIP
            embedding = embed_image(pil_image)
            all_embeddings.append(embedding)
            
            # Create document for image
            image_doc = Document(
                page_content=f"[Image: {image_id}]",
                metadata={"page": i, "type": "image", "image_id": image_id}
            )
            all_docs.append(image_doc)
            
        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")
            continue

doc.close()


In [ ]:
all_embeddings

In [ ]:
all_docs

In [ ]:
#create a vector store
#create unified FAISS vector store with CLIP embeddings
embeddings_array = np.array(all_embeddings)

#create custom FAISS index since we have precomputed embeddings
vector_store = FAISS.from_embeddings(
    text_embeddings=[(doc.page_content, emb) for doc, emb in zip(all_docs, embeddings_array)],
    embedding=None, # we are using precomputed embeddings
    metadatas=[doc.metadata for doc in all_docs]
)

In [ ]:
##Initialize GPT-4 Vision Model
llm = init_chat_model("openai:gpt-4.1")
llm

In [ ]:
def retrieval_multimodal(query, k=5)
    """Unified retrieval using CLIP embeddings for both text and images"""
    #Embed query using CLIP
    query_embedding = embed_text(query)

    #search based on the query embeddings
    results = vector_store.similarity_search_by_vector(
        embeddings=query_embedding,
        k=k
    )
    return results

In [ ]:
def create_multimodal_message(query, retrieved_docs):
    """Create a message with both text and images for GPT-4V."""
    content = []
    
    # Add the query
    content.append({
        "type": "text",
        "text": f"Question: {query}\n\nContext:\n"
    })
    
    # Separate text and image documents
    text_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "text"]
    image_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "image"]
    
    # Add text context
    if text_docs:
        text_context = "\n\n".join([
            f"[Page {doc.metadata['page']}]: {doc.page_content}"
            for doc in text_docs
        ])
        content.append({
            "type": "text",
            "text": f"Text excerpts:\n{text_context}\n"
        })
    
    # Add images
    for doc in image_docs:
        image_id = doc.metadata.get("image_id")
        if image_id and image_id in image_data_store:
            content.append({
                "type": "text",
                "text": f"\n[Image from page {doc.metadata['page']}]:\n"
            })
            content.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{image_data_store[image_id]}"
                }
            })
    
    # Add instruction
    content.append({
        "type": "text",
        "text": "\n\nPlease answer the question based on the provided text and images."
    })
    
    return HumanMessage(content=content)

In [ ]:
def multimodal_pdf_rag_pipeline(query):
    """Main pipeline for multimodal RAG."""
    # Retrieve relevant documents
    context_docs = retrieve_multimodal(query, k=5)
    
    # Create multimodal message
    message = create_multimodal_message(query, context_docs)
    
    # Get response from GPT-4V
    response = llm.invoke([message])
    
    # Print retrieved context info
    print(f"\nRetrieved {len(context_docs)} documents:")
    for doc in context_docs:
        doc_type = doc.metadata.get("type", "unknown")
        page = doc.metadata.get("page", "?")
        if doc_type == "text":
            preview = doc.page_content[:100] + "..." if len(doc.page_content) > 100 else doc.page_content
            print(f"  - Text from page {page}: {preview}")
        else:
            print(f"  - Image from page {page}")
    print("\n")
    
    return response.content

In [ ]:
if __name__ == "__main__":
    # Example queries
    queries = [
        "What does the chart on page 1 show about revenue trends?",
        "Summarize the main findings from the document",
        "What visual elements are present in the document?"
    ]
    
    for query in queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        answer = multimodal_pdf_rag_pipeline(query)
        print(f"Answer: {answer}")
        print("=" * 70)